this is to test the tftlite model 


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os

# === CONFIG ===
MODEL_PATH = '/home/admin1/Documents/rohit/RRSA/RoadNetApp/mobile/assets/roadnet_v4.tflite'         # Path to your .tflite model
LABELS_PATH = '/home/admin1/Documents/rohit/RRSA/RoadNetApp/mobile/assets/labels.txt'               # Path to label file (one label per line)
IMAGE_DIR = '/home/admin1/Documents/rohit/RRSA/RoadNetApp/mobile/assets/images/'               # Folder containing test images
OUTPUT_DIR = '/home/admin1/Documents/rohit/RRSA/RoadNetApp/mobile/assets/animages/'        # Folder to save annotated images
INPUT_SIZE = (640, 640)                  # Model input size (width, height)
CONFIDENCE_THRESHOLD = 0.5               # Minimum confidence to draw box

# === LOAD LABELS ===
with open(LABELS_PATH, 'r') as f:
    LABELS = [line.strip() for line in f.readlines()]
ROAD_LABEL_INDEX = LABELS.index('Road') if 'Road' in LABELS else -1

# === LOAD TFLITE MODEL ===
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# === PREPROCESS IMAGE ===
def preprocess_image(image):
    resized = cv2.resize(image, INPUT_SIZE)
    normalized = resized.astype(np.float32) / 255.0
    input_tensor = np.expand_dims(normalized, axis=0)
    return input_tensor

# === POSTPROCESS OUTPUT ===
def parse_output(output_data, image_shape):
    boxes = []
    img_h, img_w = image_shape[:2]
    num_boxes = 8400
    num_values = 234

    for i in range(num_boxes):
        offset = i * num_values
        objectness = output_data[offset + 4]
        if objectness < 0.3:
            continue

        score = output_data[offset + 5 + ROAD_LABEL_INDEX]
        confidence = objectness * score
        if confidence > CONFIDENCE_THRESHOLD:
            x = output_data[offset + 0] * img_w
            y = output_data[offset + 1] * img_h
            w = output_data[offset + 2] * img_w
            h = output_data[offset + 3] * img_h

            left = int(x - w / 2)
            top = int(y - h / 2)
            right = int(x + w / 2)
            bottom = int(y + h / 2)

            boxes.append({
                'label': 'Road',
                'confidence': confidence,
                'box': [left, top, right, bottom]
            })
    return boxes

# === DRAW BOXES ===
def draw_boxes(image, boxes):
    for box in boxes:
        left, top, right, bottom = box['box']
        cv2.rectangle(image, (left, top), (right, bottom), (0, 255, 0), 2)
        label = f"{box['label']} {box['confidence']:.2f}"
        cv2.putText(image, label, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return image

# === MAIN LOOP ===
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for filename in image_files:
    image_path = os.path.join(IMAGE_DIR, filename)
    image = cv2.imread(image_path)
    if image is None:
        print(f"⚠️ Failed to load {filename}")
        continue

    input_tensor = preprocess_image(image)
    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index']).flatten()

    boxes = parse_output(output_data, image.shape)
    annotated = draw_boxes(image.copy(), boxes)

    output_path = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(output_path, annotated)
    print(f"✅ Saved: {output_path} ({len(boxes)} detections)")
